In [2]:
!nvidia-smi


Sun Mar 29 10:07:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!git clone https://github.com/saccharomycetes/mllms_know.git
%cd /content/mllms_know


Cloning into 'mllms_know'...
remote: Enumerating objects: 5371, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 5371 (delta 30), reused 27 (delta 27), pack-reused 5335 (from 2)
Receiving objects: 100% (5371/5371), 32.38 MiB | 27.02 MiB/s, done.
Resolving deltas: 100% (1392/1392), done.
/content/mllms_know


In [4]:
!pip install -r requirements.txt


In [5]:
%cd /content/mllms_know/transformers
!pip install -e .
%cd /content/mllms_know


/content/mllms_know/transformers
Obtaining file:///content/mllms_know/transformers
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 102.1 MB/s eta 0:00:00
  Building editable for transformers (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-4.47.0.dev0-0.editable-py3-none-any.whl size=17428 sha256=e660164dc41094e6c854941858008d8f54e67fbb3ddf796333f44f2b1a1ce9b2
  Stored in directory: /tmp/pip-ephem-wheel-cache-x4zup2ga/wheels/1c/22/58/56e810460716a81d9d431990217451c4330ac0196c9d94c6b4
Successfully built transformers
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfu

In [6]:
!mkdir -p /content/mllms_know/data/textvqa/images
!wget https://dl.fbaipublicfiles.com/textvqa/images/train_val_images.zip -P /content/mllms_know/data/textvqa/images
!unzip -q /content/mllms_know/data/textvqa/images/train_val_images.zip -d /content/mllms_know/data/textvqa/images
!rm /content/mllms_know/data/textvqa/images/train_val_images.zip
!mv /content/mllms_know/data/textvqa/images/train_images/* /content/mllms_know/data/textvqa/images/
!rm -r /content/mllms_know/data/textvqa/images/train_images
!wget https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json -P /content/mllms_know/data/textvqa


--2026-03-29 10:08:08--  https://dl.fbaipublicfiles.com/textvqa/images/train_val_images.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.163.189.14, 3.163.189.51, 3.163.189.96, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.163.189.14|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7072297970 (6.6G) [application/zip]
Saving to: ‘/content/mllms_know/data/textvqa/images/train_val_images.zip’

train_val_images.zi 100%[===================>]   6.59G  64.9MB/s    in 88s     

2026-03-29 10:09:36 (76.4 MB/s) - ‘/content/mllms_know/data/textvqa/images/train_val_images.zip’ saved [7072297970/7072297970]

--2026-03-29 10:10:59--  https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.163.24.87, 3.163.24.72, 3.163.24.51, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.163.24.87|:443... connected.
HTTP request sent, awaiting response... 200

In [7]:
import json

with open('/content/mllms_know/data/textvqa/TextVQA_0.5.1_val.json') as f:
    datas = json.load(f)

new_datas = []
for data_id, data in enumerate(datas['data']):
    data_id = str(data_id).zfill(10)
    question = data['question']
    labels = data['answers']
    image_path = f"{data['image_id']}.jpg"
    new_data = {
        'id': data_id,
        'question': question,
        'labels': labels,
        'image_path': image_path
    }
    new_datas.append(new_data)

with open('/content/mllms_know/data/textvqa/data.json', 'w') as f:
    json.dump(new_datas, f, indent=4)

print("Total examples:", len(new_datas))


Total examples: 5000


In [8]:
import json
import shutil

full_path = "/content/mllms_know/data/textvqa/data.json"
backup_path = "/content/mllms_know/data/textvqa/data_full_backup.json"

shutil.copy(full_path, backup_path)

with open(full_path) as f:
    full_data = json.load(f)

small_data = full_data[:1]

with open(full_path, "w") as f:
    json.dump(small_data, f, indent=4)

print("Smoke-test examples:", len(small_data))
print(small_data[0])


Smoke-test examples: 1
{'id': '0000000000', 'question': 'what is the brand of this camera?', 'labels': ['nous les gosses', 'dakota', 'clos culombu', 'dakota digital', 'dakota', 'dakota', 'dakota digital', 'dakota digital', 'dakota', 'dakota'], 'image_path': '003a8ae2ef43b901.jpg'}


In [9]:
from pathlib import Path
import re

# Fix run.py by replacing everything from the top through the argparse import block
run_path = Path("/content/mllms_know/run.py")
run_text = run_path.read_text()

run_text = re.sub(
    r"^import os.*?import argparse\n",
    """import os
from PIL import Image
import torch
import numpy as np
from transformers import AutoProcessor, LlavaForConditionalGeneration, InstructBlipProcessor, InstructBlipForConditionalGeneration

try:
    from transformers import Qwen2_5_VLForConditionalGeneration
except ImportError:
    Qwen2_5_VLForConditionalGeneration = None

import argparse
""",
    run_text,
    flags=re.S,
)

run_path.write_text(run_text)

# Fix utils.py by replacing everything from the top through the base64 import block
utils_path = Path("/content/mllms_know/utils.py")
utils_text = utils_path.read_text()

utils_text = re.sub(
    r"^import torchvision\.transforms\.functional as TF.*?import base64\n",
    """import torchvision.transforms.functional as TF
import numpy as np
import os
from scipy.ndimage import median_filter
from skimage.measure import block_reduce

try:
    from qwen_vl_utils import process_vision_info
except ImportError:
    process_vision_info = None

from io import BytesIO
import base64
""",
    utils_text,
    flags=re.S,
)

utils_path.write_text(utils_text)

print("Cleanly rewrote import sections in run.py and utils.py")


Cleanly rewrote import sections in run.py and utils.py


In [10]:
with open("/content/mllms_know/utils.py") as f:
    for i in range(15):
        print(f.readline(), end="")


import torchvision.transforms.functional as TF
import numpy as np
import os
from scipy.ndimage import median_filter
from skimage.measure import block_reduce

try:
    from qwen_vl_utils import process_vision_info
except ImportError:
    process_vision_info = None

from io import BytesIO
import base64

def encode_base64(image):


In [11]:
!pip install bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 66.3 MB/s eta 0:00:00


In [12]:
from pathlib import Path

path = Path("/content/mllms_know/run.py")
text = path.read_text()

old = """    if args.model == 'llava':
        model = LlavaForConditionalGeneration.from_pretrained(args.model_id, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, attn_implementation="eager").to(args.device)
        processor = AutoProcessor.from_pretrained(args.model_id)"""

new = """    if args.model == 'llava':
        from transformers import BitsAndBytesConfig
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        model = LlavaForConditionalGeneration.from_pretrained(
            args.model_id,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            attn_implementation="eager",
            quantization_config=quantization_config,
            device_map="auto",
        )
        processor = AutoProcessor.from_pretrained(args.model_id)"""

text = text.replace(old, new)
path.write_text(text)

print("Patched run.py for 4-bit LLaVA loading")


Patched run.py for 4-bit LLaVA loading


In [13]:
!mkdir -p /content/mllms_know/data/results
!python run.py --task textvqa --model llava --method grad_att --save_path /content/mllms_know/data/results



2026-03-29 10:13:53.291325: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774779233.512496    6087 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774779233.591177    6087 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774779234.071361    6087 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774779234.071407    6087 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774779234.071411    6087 computation_placer.cc:177] computation placer alr

In [14]:
!ls -lh /content/mllms_know/data/results


total 4.0K
-rw-r--r-- 1 root root 638 Mar 29 10:18 llava-textvqa-grad_att.json


In [15]:
!python get_score.py --data_dir /content/mllms_know/data/results --save_path /content/mllms_know


/content/mllms_know/get_score.py:34: SyntaxWarning: invalid escape sequence '\d'
  periodStrip  = re.compile("(?!<=\d)(\.)(?!\d)")
/content/mllms_know/get_score.py:35: SyntaxWarning: invalid escape sequence '\d'
  commaStrip   = re.compile("(\d)(\,)(\d)")
100% 1/1 [00:00<00:00, 1470.14it/s]
  model_name     task    method  raw_acc  crop_acc
0      llava  textvqa  grad_att     90.0      90.0


In [16]:
import json

with open("/content/mllms_know/evaluation_report.json") as f:
    report = json.load(f)

[x for x in report if x["model_name"] == "llava" and x["task"] == "textvqa" and x["method"] == "grad_att"]


[{'model_name': 'llava',
  'task': 'textvqa',
  'method': 'grad_att',
  'raw_acc': 89.99999999999999,
  'crop_acc': 89.99999999999999}]

Trying with 3 Examples now

In [17]:
import shutil

shutil.copy(
    "/content/mllms_know/data/textvqa/data_full_backup.json",
    "/content/mllms_know/data/textvqa/data.json"
)
print("Full dataset restored")


Full dataset restored


In [18]:
import json

full_path = "/content/mllms_know/data/textvqa/data.json"

with open(full_path) as f:
    full_data = json.load(f)

small_data = full_data[:3]

with open(full_path, "w") as f:
    json.dump(small_data, f, indent=4)

print("Now testing grad_att with", len(small_data), "examples")
print(small_data[0])


Now testing grad_att with 3 examples
{'id': '0000000000', 'question': 'what is the brand of this camera?', 'labels': ['nous les gosses', 'dakota', 'clos culombu', 'dakota digital', 'dakota', 'dakota', 'dakota digital', 'dakota digital', 'dakota', 'dakota'], 'image_path': '003a8ae2ef43b901.jpg'}


In [19]:
!mkdir -p /content/mllms_know/data/results
!rm -f /content/mllms_know/data/results/llava-textvqa-grad_att.json


In [20]:
!python run.py --task textvqa --model llava --method grad_att --save_path /content/mllms_know/data/results


2026-03-29 10:25:54.280549: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774779954.307058    9333 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774779954.315743    9333 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774779954.344828    9333 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774779954.344871    9333 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774779954.344876    9333 computation_placer.cc:177] computation placer alr

In [21]:
!python get_score.py --data_dir /content/mllms_know/data/results --save_path /content/mllms_know


/content/mllms_know/get_score.py:34: SyntaxWarning: invalid escape sequence '\d'
  periodStrip  = re.compile("(?!<=\d)(\.)(?!\d)")
/content/mllms_know/get_score.py:35: SyntaxWarning: invalid escape sequence '\d'
  commaStrip   = re.compile("(\d)(\,)(\d)")
100% 3/3 [00:00<00:00, 1793.46it/s]
  model_name     task    method  raw_acc  crop_acc
0      llava  textvqa  grad_att     30.0      40.0


In [22]:
import shutil

shutil.copy(
    "/content/mllms_know/data/textvqa/data_full_backup.json",
    "/content/mllms_know/data/textvqa/data.json"
)
print("Full dataset restored")


Full dataset restored


In [23]:
import json

with open("/content/mllms_know/data/textvqa/data.json") as f:
    full_data = json.load(f)

small_data = full_data[:5]

with open("/content/mllms_know/data/textvqa/data.json", "w") as f:
    json.dump(small_data, f, indent=4)

print("Now testing grad_att with", len(small_data), "examples")


Now testing grad_att with 5 examples


In [24]:
!rm -f /content/mllms_know/data/results/llava-textvqa-grad_att.json
!python run.py --task textvqa --model llava --method grad_att --save_path /content/mllms_know/data/results


2026-03-29 10:30:26.053959: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774780226.081316   10573 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774780226.090952   10573 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774780226.124043   10573 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774780226.124077   10573 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774780226.124080   10573 computation_placer.cc:177] computation placer alr

In [25]:
!python get_score.py --data_dir /content/mllms_know/data/results --save_path /content/mllms_know


/content/mllms_know/get_score.py:34: SyntaxWarning: invalid escape sequence '\d'
  periodStrip  = re.compile("(?!<=\d)(\.)(?!\d)")
/content/mllms_know/get_score.py:35: SyntaxWarning: invalid escape sequence '\d'
  commaStrip   = re.compile("(\d)(\,)(\d)")
100% 5/5 [00:00<00:00, 1839.77it/s]
  model_name     task    method  raw_acc  crop_acc
0      llava  textvqa  grad_att     58.0      64.0


In [26]:
import shutil

shutil.copy(
    "/content/mllms_know/data/textvqa/data_full_backup.json",
    "/content/mllms_know/data/textvqa/data.json"
)
print("Full dataset restored")


Full dataset restored


In [27]:
import json

with open("/content/mllms_know/data/textvqa/data.json") as f:
    full_data = json.load(f)

small_data = full_data[:10]

with open("/content/mllms_know/data/textvqa/data.json", "w") as f:
    json.dump(small_data, f, indent=4)

print("Now testing grad_att with", len(small_data), "examples")
print(small_data[0])


Now testing grad_att with 10 examples
{'id': '0000000000', 'question': 'what is the brand of this camera?', 'labels': ['nous les gosses', 'dakota', 'clos culombu', 'dakota digital', 'dakota', 'dakota', 'dakota digital', 'dakota digital', 'dakota', 'dakota'], 'image_path': '003a8ae2ef43b901.jpg'}


In [28]:
!mkdir -p /content/mllms_know/data/results
!rm -f /content/mllms_know/data/results/llava-textvqa-grad_att.json


In [29]:
!python run.py --task textvqa --model llava --method grad_att --save_path /content/mllms_know/data/results


2026-03-29 10:35:38.825388: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774780538.853192   12000 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774780538.866163   12000 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774780538.898489   12000 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774780538.898521   12000 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774780538.898527   12000 computation_placer.cc:177] computation placer alr

In [30]:
!python get_score.py --data_dir /content/mllms_know/data/results --save_path /content/mllms_know


/content/mllms_know/get_score.py:34: SyntaxWarning: invalid escape sequence '\d'
  periodStrip  = re.compile("(?!<=\d)(\.)(?!\d)")
/content/mllms_know/get_score.py:35: SyntaxWarning: invalid escape sequence '\d'
  commaStrip   = re.compile("(\d)(\,)(\d)")
100% 10/10 [00:00<00:00, 1844.79it/s]
  model_name     task    method  raw_acc  crop_acc
0      llava  textvqa  grad_att     39.0      55.0


In [31]:
import json

with open("/content/mllms_know/evaluation_report.json") as f:
    report = json.load(f)

[x for x in report if x["model_name"] == "llava" and x["task"] == "textvqa" and x["method"] == "grad_att"]


[{'model_name': 'llava',
  'task': 'textvqa',
  'method': 'grad_att',
  'raw_acc': 39.0,
  'crop_acc': 55.0}]

## **Pure Grad Method**

In [42]:
import shutil

shutil.copy(
    "/content/mllms_know/data/textvqa/data_full_backup.json",
    "/content/mllms_know/data/textvqa/data.json"
)
print("Full dataset restored")


Full dataset restored


In [43]:
import json

with open("/content/mllms_know/data/textvqa/data.json") as f:
    full_data = json.load(f)

small_data = full_data[:3]

with open("/content/mllms_know/data/textvqa/data.json", "w") as f:
    json.dump(small_data, f, indent=4)

print("Now testing pure_grad with", len(small_data), "examples")
print(small_data[0])


Now testing pure_grad with 3 examples
{'id': '0000000000', 'question': 'what is the brand of this camera?', 'labels': ['nous les gosses', 'dakota', 'clos culombu', 'dakota digital', 'dakota', 'dakota', 'dakota digital', 'dakota digital', 'dakota', 'dakota'], 'image_path': '003a8ae2ef43b901.jpg'}


In [44]:
!mkdir -p /content/mllms_know/data/results
!rm -f /content/mllms_know/data/results/llava-textvqa-grad_att.json
!rm -f /content/mllms_know/data/results/llava-textvqa-pure_grad.json


In [45]:
!python run.py --task textvqa --model llava --method pure_grad --save_path /content/mllms_know/data/results


2026-03-29 10:56:32.105651: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774781792.132102   17568 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774781792.141796   17568 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774781792.172922   17568 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774781792.172951   17568 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774781792.172954   17568 computation_placer.cc:177] computation placer alr

In [47]:

import os
print(os.path.exists("/content/mllms_know/data/results/llava-textvqa-pure_grad.json"))


False


In [38]:
import json

with open("/content/mllms_know/evaluation_report.json") as f:
    report = json.load(f)

[x for x in report if x["model_name"] == "llava" and x["task"] == "textvqa" and x["method"] == "pure_grad"]


[]

In [39]:
!ls -lh /content/mllms_know/data/results


total 8.0K
-rw-r--r-- 1 root root 6.0K Mar 29 10:37 llava-textvqa-grad_att.json


In [40]:
import os
print(os.path.exists("/content/mllms_know/data/results/llava-textvqa-pure_grad.json"))


False
